# 지하철 혼잡도 분석 시안 1

## 대주제
역별 이용량이 많을수록 혼잡도도 높은가?

## 분석 질문
- 이용량이 많은 역은 평균 혼잡도도 높은가?
- 이용량과 혼잡도가 모두 높은 역은 어디인가?
- 각 역의 최고 혼잡 시간대는 언제인가?

## 데이터 범위와 한계
- df1: 2026년 7월 역별 승하차 이용량
- df4: 최근 3개월 역별 시간대별 혼잡도
- 두 데이터의 기간이 다르므로 결과는 동일 기간의 인과관계가 아닌 탐색적 관계로 해석한다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
df1 = pd.read_csv(
    '../data/실습데이터/2_CARD_SUBWAY_MONTH_202607.csv',
    encoding='utf-8-sig',
    index_col=False
)

df4 = pd.read_csv(
    '../data/실습데이터/5_서울교통공사_지하철혼잡도정보.csv',
    encoding='cp949'
)

display(df1.head())
display(df4.head())

## 1. df1 이용량 정제 및 역별 집계

df1의 첫 행이 `사용일자` 형식의 8자리 날짜인지 확인한다. 날짜가 노선명으로 보이면 원본 CSV를 먼저 복구해야 한다.

In [ ]:
expected_cols = ['사용일자', '노선명', '역명', '승차총승객수', '하차총승객수', '등록일자']
print(df1.columns.tolist())
print(df1.shape)

date_check = df1['사용일자'].astype('string').str.fullmatch(r'\d{8}', na=False)
print('사용일자 형식 정상 비율:', date_check.mean())

for col in ['승차총승객수', '하차총승객수']:
    df1[col] = pd.to_numeric(df1[col], errors='coerce')

df1['이용량'] = df1['승차총승객수'] + df1['하차총승객수']

usage_summary = (
    df1.groupby(['노선명', '역명'], as_index=False)
       .agg(
           승차인원=('승차총승객수', 'sum'),
           하차인원=('하차총승객수', 'sum'),
           이용량=('이용량', 'sum')
       )
       .sort_values('이용량', ascending=False)
)

usage_summary.head(10)

## 2. df4 시간대별 혼잡도 정제

df4의 앞 5개 컬럼은 역 정보이고, 6번째 컬럼부터 시간대별 혼잡도라고 가정한다.

In [ ]:
# 실제 컬럼 위치를 먼저 확인
print(df4.columns.tolist())

# 파일 구조 기준으로 공통 키 이름을 통일
df4 = df4.rename(columns={
    df4.columns[1]: '호선',
    df4.columns[3]: '역명'
})

id_cols = df4.columns[:5].tolist()
time_cols = df4.columns[5:].tolist()

df4_long = df4.melt(
    id_vars=id_cols,
    value_vars=time_cols,
    var_name='시간대',
    value_name='혼잡도'
)
df4_long['혼잡도'] = pd.to_numeric(df4_long['혼잡도'], errors='coerce')

time_summary = (
    df4_long.groupby(['호선', '역명', '시간대'], as_index=False)
           .agg(평균혼잡도=('혼잡도', 'mean'))
)

congestion_summary = (
    time_summary.groupby(['호선', '역명'], as_index=False)
                .agg(
                    평균혼잡도=('평균혼잡도', 'mean'),
                    최대혼잡도=('평균혼잡도', 'max')
                )
)

congestion_summary.head()

In [ ]:
peak_idx = time_summary.groupby(['호선', '역명'])['평균혼잡도'].idxmax()
peak_time_df = time_summary.loc[peak_idx].reset_index(drop=True)
peak_time_df.head()

## 3. 이용량과 혼잡도 결합

노선명과 역명을 기준으로 집계된 두 데이터를 결합한다.

In [ ]:
analysis_df = usage_summary.merge(
    congestion_summary,
    left_on=['노선명', '역명'],
    right_on=['호선', '역명'],
    how='inner'
)

usage_median = analysis_df['이용량'].median()
congestion_median = analysis_df['평균혼잡도'].median()

analysis_df['이용량구분'] = np.where(
    analysis_df['이용량'] >= usage_median, '높음', '낮음'
)
analysis_df['혼잡도구분'] = np.where(
    analysis_df['평균혼잡도'] >= congestion_median, '높음', '낮음'
)

analysis_df['분석유형'] = (
    '이용량 ' + analysis_df['이용량구분'] +
    '· 혼잡도 ' + analysis_df['혼잡도구분']
)

insight_df = analysis_df.merge(
    peak_time_df,
    on=['호선', '역명'],
    how='left',
    suffixes=('', '_최고시간대')
)

insight_df.head()

In [ ]:
result_cols = [
    '노선명', '역명', '이용량', '평균혼잡도',
    '분석유형', '시간대', '평균혼잡도_최고시간대'
]

insight_df[result_cols].sort_values(
    by=['분석유형', '평균혼잡도_최고시간대'],
    ascending=[True, False]
).head(20)

## 4. 관계 확인과 인사이트

- 이용량 높음·혼잡도 높음: 집중 관리가 필요한 역
- 이용량 높음·혼잡도 낮음: 이용객은 많지만 상대적으로 수용력이 좋은 역
- 이용량 낮음·혼잡도 높음: 이용량 대비 혼잡한 역
- 이용량 낮음·혼잡도 낮음: 상대적으로 여유로운 역

`df1`은 2026년 7월, `df4`는 최근 3개월 자료이므로 결과는 탐색적 관계로 해석한다.

In [ ]:
print(analysis_df[['이용량', '평균혼잡도']].corr())

sns.scatterplot(
    data=analysis_df,
    x='이용량',
    y='평균혼잡도',
    hue='분석유형'
)
plt.title('역별 이용량과 평균 혼잡도의 관계')
plt.xlabel('2026년 7월 이용량')
plt.ylabel('최근 3개월 평균 혼잡도')
plt.show()